# Yahoo Finance Causality Demo

This notebook uses only functions from `src` to load two one-dimensional time series, preprocess them, and run causal analysis.

The default example uses Nasdaq vs S&P 500, and you can switch the tickers to SPY vs VIX or any other pair of 1-D series.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists() and (project_root.parent / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from src import run_yfinance_demo

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [ ]:
# Example defaults: Nasdaq and S&P 500. Change these values before running if desired.
ticker_one = '^IXIC'
ticker_two = '^GSPC'
start = '2018-01-01'
end = '2024-12-31'
name_one = None
name_two = None

(Date
 2018-01-02    2695.810059
 2018-01-03    2713.060059
 2018-01-04    2723.989990
 2018-01-05    2743.149902
 2018-01-08    2747.709961
 Name: NASDAQ Composite, dtype: float64,
 Date
 2018-01-02    7006.899902
 2018-01-03    7065.529785
 2018-01-04    7077.910156
 2018-01-05    7136.560059
 2018-01-08    7157.390137
 Name: S&P 500, dtype: float64)

In [ ]:
run_yfinance_demo(ticker_one=ticker_one, ticker_two=ticker_two, start=start, end=end, name_one=name_one, name_two=name_two)

## Step-by-step workflow

The cells below show the pipeline one step at a time, then finish with the one-call convenience function for a quick demo.

### 1. Load a pair of series

The raw data load is separated from the later transforms so each step can be shown clearly.

In [ ]:
from src import PipelineConfig, PreprocessingConfig, AnalysisConfig, download_yfinance_series, preprocess_series_pair, adf_unit_root_test, run_analysis_pipeline, run_full_analysis_for_pair

# Pick any pair you want to demo.
ticker_one = "^IXIC"
ticker_two = "^GSPC"
start = "2018-01-01"
end = "2024-12-31"

loaded = download_yfinance_series(ticker_one, ticker_two, start, end)
loaded.left.head(), loaded.right.head()

### 2. Preprocess the series

Use a small config object to control the base representation and optional transforms.

In [ ]:
preprocessing_config = PreprocessingConfig(
    base_representation="log_returns",
    smoothing_window=5,
    standardize=False,
)

preprocessed = preprocess_series_pair(
    loaded.left,
    loaded.right,
    loaded.left_name,
    loaded.right_name,
    preprocessing_config,
)

preprocessed.left_label, preprocessed.right_label, preprocessed.operations

### 3. Check stationarity

The ADF test tells you whether differencing or another transform is needed before the causal metrics.

In [ ]:
stationarity_left = adf_unit_root_test(preprocessed.left, preprocessed.left_label)
stationarity_right = adf_unit_root_test(preprocessed.right, preprocessed.right_label)

stationarity_left, stationarity_right

### 4. Run the full analysis pipeline

This returns the structured outputs so you can inspect DTW, Granger, TE, CCM, and surrogate results in separate notebook cells.

In [ ]:
analysis_config = AnalysisConfig(
    run_lagged_cross_correlation=True,
    run_dtw=True,
    run_granger=True,
    run_granger_surrogates=False,
    run_te=True,
    run_te_surrogates=False,
    run_ccm=True,
    run_ccm_convergence=False,
    run_ccm_surrogates=False,
)

results = run_analysis_pipeline(
    loaded.left,
    loaded.right,
    loaded.left_name,
    loaded.right_name,
    preprocessing_config=preprocessing_config,
    analysis_config=analysis_config,
)

results.keys()

### 5. One-call full demo

When you want the shortest possible demonstration, use the bundled convenience function.

In [ ]:
quick_demo_config = PipelineConfig(
    preprocessing=preprocessing_config,
    analysis=analysis_config,
)

run_full_analysis_for_pair(
    ticker_one=ticker_one,
    ticker_two=ticker_two,
    config=quick_demo_config,
    start=start,
    end=end,
    name_one=loaded.left_name,
    name_two=loaded.right_name,
)